# 02 · Suy luận (inference)

Chạy pipeline `datamate` trên một split và ghi `submission.jsonl`: nạp shop → view và lớp công nợ → 8 mẫu SQL thường + 4 mẫu thinking cho câu
khó → chạy, sửa lỗi, bỏ phiếu. Cờ lấy từ `scripts/flags.sh` (cấu hình của dự án) và tự chỉnh theo GPU.

**Phần cứng:** một GPU bf16 (Ampere trở lên) từ 24 GB cho mỗi tiến trình. Nhiều GPU thì mỗi GPU một phần shop. GPU Ada / Hopper / Blackwell
chạy fp8 như cấu hình dự án, Ampere (A10, A100…) chạy bf16 với context ngắn hơn. T4 / P100 không chạy được model này.

**Thời gian:** model nạp vài phút, `LIMIT = 10` để thử, cả 1000 câu public khoảng 3,5 giờ trên một GPU 24 GB.

**Dùng cho train:** `SPLIT = "train"`, `REUSE_LAYERS = False`, `SAVE_PROMPTS = True` cho ra lượt chạy có nhãn mà `04_training_data` cần.

In [ ]:
# Setup. Local checkout: nothing to install. Kaggle: Internet on, GPU accelerator for inference / training, the competition data
# attached as a private dataset, and the code either attached as a dataset (a copy of the checkout) or cloned with your
# GitHub token in Add-ons -> Secrets as GITHUB_TOKEN (if the repo is private). Colab: Google Drive holding MyDrive/datamate/code and
# MyDrive/datamate/data (or a GITHUB_TOKEN secret for the code), a GPU runtime for inference / training.
import os, shutil, subprocess, sys
from pathlib import Path

REPO_URL = "github.com/hana200804/datamate-agent-Locked-In1.git"


def clone(root, token):
    subprocess.run(["git", "clone", "--depth", "1", f"https://x-access-token:{token}@{REPO_URL}", str(root)], check=True)
    subprocess.run(["git", "remote", "set-url", "origin", f"https://{REPO_URL}"], cwd=root, check=True)  # token out of saved files


if Path("/kaggle/input").exists():
    root = Path("/tmp/datamate-agent-Locked-In1")
    attached = sorted(p.parent.parent for p in Path("/kaggle/input").glob("**/datamate/notebook.py"))
    if not root.exists() and attached:
        shutil.copytree(attached[0], root)                         # code attached as a dataset: no token needed
    if not root.exists():
        from kaggle_secrets import UserSecretsClient
        clone(root, UserSecretsClient().get_secret("GITHUB_TOKEN"))
elif "COLAB_RELEASE_TAG" in os.environ:
    from google.colab import drive
    drive.mount("/content/drive")
    root, code = Path("/content/datamate-agent-Locked-In1"), Path("/content/drive/MyDrive/datamate/code")
    if not root.exists() and (code / "datamate/notebook.py").exists():
        shutil.copytree(code, root)                                # code uploaded to Drive: no token needed
    if not root.exists():
        from google.colab import userdata                          # key icon -> secret GITHUB_TOKEN with notebook access
        clone(root, userdata.get("GITHUB_TOKEN"))
else:
    root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
os.chdir(root)
sys.path.insert(0, str(root))
from datamate.notebook import install, setup
install("llm")
env = setup()

In [ ]:
import json
from collections import Counter

import matplotlib.pyplot as plt
import pandas as pd

read = lambda p: [json.loads(l) for l in open(p, encoding="utf-8-sig") if l.strip()]
pd.set_option("display.max_colwidth", 80)

In [ ]:
SPLIT = "public_test"        # public_test | train | private_test (a folder of the data dir)
LIMIT = 10                   # first N questions (None = all)
IDS = None                   # a file of question ids, e.g. "splits/rft_train24.txt" or "splits/holdout600.txt"
VIEWS_FROM = None           # optional path to layers from your own earlier run
LORA = None                # base model; set a path to your own trained adapter to enable LoRA
SAVE_PROMPTS = True          # keep each question's prompt in the trace (needed to build SFT rows from a train run)
OUT_NAME = "nb_" + SPLIT

In [ ]:
out = env.runs / OUT_NAME
if env.can_run_llm:
    model = env.base_model()
    lora_flags = ["--lora", env.adapter(LORA), "--lora-thinking"] if LORA else []
    flags, engine = env.inference_flags()
    n = len(env.gpus)
    layers = ["--views-from", VIEWS_FROM] if VIEWS_FROM else []
    pick = (["--limit", str(LIMIT)] if LIMIT else []) + (["--ids", IDS] if IDS else []) + (["--save-prompts"] if SAVE_PROMPTS else [])
    make_cmd = lambda i: [env.python, "-m", "datamate.predict", "--data", env.data / SPLIT, "--output", out / f"shard{i}", "--shard", f"{i}/{n}",
                          "--model", model, *lora_flags, "--fewshot", "assets/fewshot_bank.jsonl", *flags, "--engine", engine, *layers, *pick]
    print(f"{n} GPU process(es) -> {out}\nflags: {' '.join(flags)}")
else:   # Kaggle T4 / P100 or a CPU session: the same run made on a 24 GB GPU (public_test, LIMIT = 10)
    env.use_saved(OUT_NAME, out)

In [ ]:
if env.can_run_llm:
    env.run_shards(make_cmd, n, out)

In [ ]:
from datamate.submission import check, merge_shards
merge_shards(out, env.data / SPLIT / "questions.jsonl")
errors = check(out / "submission.jsonl", env.data / SPLIT / "questions.jsonl")
print("format OK" if not errors else errors[:5])

## Kết quả

Số câu đã trả lời theo shop, và vài câu mẫu (đáp án, tỷ lệ phiếu đồng ý). Với `SPLIT = "train"` có thêm điểm theo đáp án train.

In [ ]:
trace = [r for p in sorted(out.glob("shard*/trace.jsonl")) for r in read(p)]
df = pd.DataFrame([{"shop": r["business_id"], "answered": r["final"]["type"] != "abstain", "conf": r["conf"]} for r in trace])
display(df.groupby("shop").agg(questions=("answered", "size"), answered=("answered", "sum"), mean_conf=("conf", "mean")).round(2))
pd.DataFrame([{"question": r["question"][:90], "final": json.dumps(r["final"], ensure_ascii=False)[:80], "conf": round(r["conf"], 2)} for r in trace[:5]])

In [ ]:
if (env.data / SPLIT / "answers.jsonl").exists():
    from datamate import scoring
    gold = {a["id"]: a for a in read(env.data / SPLIT / "answers.jsonl")}
    report, _ = scoring.evaluate({r["id"]: gold[r["id"]] for r in trace}, {r["id"]: r["final"] for r in trace},
                                 {q["id"]: q for q in read(env.data / SPLIT / "questions.jsonl")})
    print({k: report[k] for k in ("N", "correct", "wrong", "abstain", "PA", "precision")})